In [ ]:
pip install scikit-learn pandas numpy torch

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# embeddings generados con ESM2
X = np.load("protein_embeddings.npy")

# targets
targets = pd.read_csv("targets.csv")

y = targets[["pTM", "ipTM"]].values

print("Embeddings:", X.shape)
print("Targets:", y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

dt = DecisionTreeRegressor(
    max_depth=20,
    random_state=42
)

dt.fit(X_train, y_train)

pred_dt = dt.predict(X_test)

mse_dt = mean_squared_error(y_test, pred_dt)

print("Decision Tree MSE:", mse_dt)

# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)

mse_rf = mean_squared_error(y_test, pred_rf)

print("Random Forest MSE:", mse_rf)

# Red neuronal profunda (PyTorch)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
train_dataset = TensorDataset(X_train_t, y_train_t)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [ ]:
class ProteinNet(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(1024, 512),
            nn.ReLU(),

            nn.Linear(512, 128),
            nn.ReLU(),

            nn.Linear(128, 2)  # pTM e ipTM
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = ProteinNet(X.shape[1]).to(device)

criterion = nn.MSELoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [ ]:
epochs = 50

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        pred = model(xb)

        loss = criterion(pred, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss {total_loss:.4f}")

In [ ]:
model.eval()

with torch.no_grad():

    preds = model(X_test_t.to(device)).cpu().numpy()

mse_nn = mean_squared_error(y_test, preds)

print("Neural Net MSE:", mse_nn)

In [ ]:
print("Decision Tree:", mse_dt)
print("Random Forest:", mse_rf)
print("Neural Net:", mse_nn)

In [ ]:
from scipy.stats import pearsonr

print("pTM correlation:", pearsonr(y_test[:,0], preds[:,0]))
print("ipTM correlation:", pearsonr(y_test[:,1], preds[:,1]))